In [47]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

# Load dataset dari Google Drive (langsung)
url = 'https://drive.google.com/uc?id=1LfQWProB0VjWN5q8bKuRIgn-stULfIRo'
df = pd.read_csv(url)

print('Shape awal:', df.shape)

# Backup data asli
original_df = df.copy()
df.head()

Shape awal: (130, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


In [48]:
# STEP 0 — Load & eksplorasi awal
df = pd.read_csv('housing_dirty.csv')

print('Shape awal:', df.shape)

df.info()
print(df.isnull().sum())
df.describe()

Shape awal: (130, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [49]:
# STEP 1 — Hapus Duplikat

# cek jumlah duplikat sebelum
print('Jumlah duplikat sebelum:', df.duplicated().sum())

# hapus duplikat
df.drop_duplicates(inplace=True)

# cek jumlah duplikat setelah
print('Jumlah duplikat setelah:', df.duplicated().sum())

# cek ukuran data setelah
print('Shape setelah hapus duplikat:', df.shape)

Jumlah duplikat sebelum: 0
Jumlah duplikat setelah: 0
Shape setelah hapus duplikat: (130, 7)


In [50]:
# STEP 2 — Normalisasi String
#kondisi
# Simpan data sebelum
kondisi_before = df['kondisi'].copy()

print('=== Sebelum normalisasi ===')
print(kondisi_before.value_counts())

# Normalisasi
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print('\n=== Sesudah normalisasi ===')
print(df['kondisi'].value_counts())

# Hitung berapa baris yang berubah
perubahan = (kondisi_before != df['kondisi']).sum()
print('\nJumlah data yang berubah:', perubahan)

=== Sebelum normalisasi ===
kondisi
baik              45
sedang            30
rusak             10
BAIK               8
baik sekali        7
Baik               5
SEDANG             4
cukup              4
bagus              4
Bagus              3
Sedang             3
Cukup              2
jelek              2
RUSAK              2
perlu renovasi     1
Name: count, dtype: int64

=== Sesudah normalisasi ===
kondisi
baik              58
sedang            37
rusak             12
bagus              7
baik sekali        7
cukup              6
jelek              2
perlu renovasi     1
Name: count, dtype: int64

Jumlah data yang berubah: 27


In [51]:

#kota
# Simpan data sebelum
kota_before = df['kota'].copy()

print('=== Sebelum normalisasi KOTA ===')
print(kota_before.value_counts())

# Normalisasi
df['kota'] = df['kota'].str.strip().str.title()

print('\n=== Sesudah normalisasi KOTA ===')
print(df['kota'].value_counts())

perubahan_kota = (kota_before != df['kota']).sum()
print('\nJumlah perubahan kota:', perubahan_kota)

=== Sebelum normalisasi KOTA ===
kota
Bandung       14
Medan         13
Makassar      12
Yogyakarta    10
Jakarta        9
Depok          7
Surabaya       7
Semarang       6
medan          5
jogja          5
yogyakarta     4
makassar       4
Jogja          3
semarang       3
YGY            3
mdn            2
sby            2
dpk            2
jakarta        2
DEPOK          2
Smg            2
MAKASSAR       2
surabaya       2
Mksr           2
JAKARTA        1
Bdg            1
depok          1
bandung        1
Bandung        1
SURABAYA       1
 Jakarta       1
Name: count, dtype: int64

=== Sesudah normalisasi KOTA ===
kota
Medan         18
Makassar      18
Bandung       16
Yogyakarta    14
Jakarta       13
Depok         10
Surabaya      10
Semarang       9
Jogja          8
Ygy            3
Sby            2
Dpk            2
Mdn            2
Smg            2
Mksr           2
Bdg            1
Name: count, dtype: int64

Jumlah perubahan kota: 44


In [52]:
# STEP 3 — Imputasi Missing Values

# Cek jumlah missing sebelum imputasi
print('Missing sebelum:')
print(df.isnull().sum())

# --- Imputasi untuk data numerik ---
# Menggunakan median karena lebih tahan terhadap outlier
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

# --- Imputasi untuk data kategorik / diskrit ---
# Menggunakan modus (nilai yang paling sering muncul)
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

# Cek jumlah missing setelah imputasi
print('\nMissing sesudah:')
print(df.isnull().sum())

Missing sebelum:
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64

Missing sesudah:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


In [56]:
# STEP 4 — Tangani Outlier (IQR Fence)
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

# tampilkan hasil
df

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000.0,baik
1,2,254.0,761.0,Medan,1.0,1995.0,bagus
2,3,249.7,895.0,Depok,1.0,1983.0,baik
3,4,49.7,178.0,Ygy,5.0,2013.0,baik
4,5,133.4,424.0,Medan,5.0,2004.0,sedang
...,...,...,...,...,...,...,...
125,126,330.2,1050.0,Semarang,6.0,1993.0,baik
126,127,193.8,655.0,Makassar,5.0,2008.0,baik
127,128,84.1,202.0,Makassar,2.0,1986.0,baik
128,129,88.0,252.0,Makassar,1.0,1986.0,bagus


In [60]:
# =========================
# STEP 5 — Validasi & Eksplorasi Akhir
# =========================

# Cek missing value (harus = 0)
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'

# Cek duplikat (harus = 0)
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

# Menampilkan ukuran dataset akhir (baris, kolom)
print('Shape akhir:', df.shape)

# =========================
# Tampilkan informasi kolom
# =========================

# Menampilkan semua nama kolom
print("\nNama kolom:", df.columns.tolist())

# Menampilkan jumlah kolom
print("Jumlah kolom:", len(df.columns))

# Menampilkan info detail dataset (tipe data, missing, dll)
print("\nInfo dataset:")
print(df.info())

# =========================
# Simpan dataset bersih
# =========================

df.to_csv('housing_clean.csv', index=False)

print("\nDataset bersih berhasil disimpan di housing_clean.csv!")

# =========================
# Tampilkan isi data (preview)
# =========================

print("\nPreview data (10 baris pertama):")
df.head(10)

Shape akhir: (130, 7)

Nama kolom: ['id', 'luas_m2', 'harga_juta', 'kota', 'kamar', 'tahun_bangun', 'kondisi']
Jumlah kolom: 7

Info dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       130 non-null    float64
 2   harga_juta    130 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         130 non-null    float64
 5   tahun_bangun  130 non-null    float64
 6   kondisi       130 non-null    object 
dtypes: float64(4), int64(1), object(2)
memory usage: 7.2+ KB
None

Dataset bersih berhasil disimpan di housing_clean.csv!

Preview data (10 baris pertama):


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,Jogja,2.0,2000.0,baik
1,2,254.0,761.0,Medan,1.0,1995.0,bagus
2,3,249.7,895.0,Depok,1.0,1983.0,baik
3,4,49.7,178.0,Ygy,5.0,2013.0,baik
4,5,133.4,424.0,Medan,5.0,2004.0,sedang
5,6,153.3,814.0,Jakarta,1.0,2006.0,sedang
6,7,114.3,655.0,Jakarta,3.0,2011.0,baik
7,8,193.8,333.0,Yogyakarta,1.0,1989.0,baik sekali
8,9,81.2,307.0,Yogyakarta,1.0,1996.0,sedang
9,10,69.1,237.0,Bandung,5.0,1980.0,baik


In [69]:
# =========================
# STEP 5 — Validasi & Eksplorasi Akhir
# =========================

# Mengecek apakah masih ada missing value (data kosong)
# Harus = 0, jika tidak maka program akan berhenti
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'

# Mengecek apakah ada data duplikat
# Harus = 0 agar data tidak ganda
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

# Menampilkan ukuran dataset (jumlah baris dan kolom)
print('Shape akhir:', df.shape)

# =========================
# Informasi Kolom Dataset
# =========================

# Menampilkan semua nama kolom
print("\nNama kolom:", df.columns.tolist())

# Menampilkan jumlah kolom dalam dataset
print("Jumlah kolom:", len(df.columns))

# Menampilkan struktur dataset (tipe data, jumlah non-null, dll)
print("\nInfo dataset:")
print(df.info())

# =========================
# Preview Data
# =========================

# Menampilkan 10 baris pertama untuk melihat isi data
print("\nPreview data (10 baris pertama):")
print(df.head(10))

# =========================
# Ekspor dataset ke CSV
# =========================

df.to_csv('housing_clean.csv', index=False)

print("Dataset berhasil diekspor sebagai 'housing_clean.csv'")

# =========================
# Akses API JSONPlaceholder dan simpan sebagai DataFrame
# =========================

import requests
import pandas as pd

# URL API
url = "https://jsonplaceholder.typicode.com/posts"

# Ambil data dari API
response = requests.get(url, timeout=10)

# Cek status request
if response.status_code == 200:

    # Ubah JSON menjadi DataFrame
    api_df = pd.DataFrame(response.json())

    # Tampilkan 5 data pertama
    print("\nData dari API (5 baris pertama):")
    print(api_df.head())

else:
    print("Error:", response.status_code)

Shape akhir: (130, 7)

Nama kolom: ['id', 'luas_m2', 'harga_juta', 'kota', 'kamar', 'tahun_bangun', 'kondisi']
Jumlah kolom: 7

Info dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       130 non-null    float64
 2   harga_juta    130 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         130 non-null    float64
 5   tahun_bangun  130 non-null    float64
 6   kondisi       130 non-null    object 
dtypes: float64(4), int64(1), object(2)
memory usage: 7.2+ KB
None

Preview data (10 baris pertama):
   id  luas_m2  harga_juta        kota  kamar  tahun_bangun      kondisi
0   1    297.0      1084.0       Jogja    2.0        2000.0         baik
1   2    254.0       761.0       Medan    1.0        1995.0        bagus
2   3    249.7       895.0       Depok    1.0  